In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/csv/processed_data.csv')


Index                 128
tire_number        411624
origin            3325157
measurement_id     411624
200.0              411624
                   ...   
999.6              411624
999.7              411624
999.8              411624
999.9              411624
1000.0             411624
Length: 8005, dtype: int64


In [3]:
print(df.memory_usage(deep=True).sum())

3297552157


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51453 entries, 0 to 51452
Columns: 8004 entries, tire_number to 1000.0
dtypes: float64(8001), int64(2), object(1)
memory usage: 3.1+ GB


In [6]:
# Multi-Spectrum Comparison - visualize multiple measurements by component (Interactive Plotly Version)
print("🔄 Creating interactive mean intensity comparison across all components and wavelengths...")

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

non_feature_cols = ['tire_number', 'origin', 'measurement_id']

numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
wavelength_columns = [col for col in numeric_columns if col not in non_feature_cols]

wavelength_values = [float(col) for col in wavelength_columns]


# Get all unique components
components = df['origin'].unique()
colors = px.colors.qualitative.Set1[:len(components)]  # Use Plotly color palette

# Calculate mean intensity for each component across all wavelengths
component_means = {}
component_stds = {}

for component in components:
    component_data = df[df['origin'] == component]
    
    # Calculate mean intensity for each wavelength across all tires for this component
    mean_intensities = []
    std_intensities = []
    
    for wl_col in wavelength_columns:
        intensities = component_data[wl_col].values
        mean_intensities.append(np.mean(intensities))
        std_intensities.append(np.std(intensities))
    
    component_means[component] = mean_intensities
    component_stds[component] = std_intensities

# Create subplots with Plotly
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Mean LIBS Spectra Comparison - All Components from all tires', 
                   'Mean LIBS Spectra with Standard Deviation Bands from all tires'),
    horizontal_spacing=0.1
)

# Plot 1: Mean intensity spectra for all components
for i, component in enumerate(components):
    color = colors[i % len(colors)]
    
    fig.add_trace(
        go.Scatter(
            x=wavelength_values,
            y=component_means[component],
            mode='lines',
            name=f'{component.title()}',
            line=dict(color=color, width=2),
            opacity=0.8,
            hovertemplate='<b>%{fullData.name}</b><br>' +
                         'Wavelength: %{x:.1f} nm<br>' +
                         'Mean Intensity: %{y:.2f}<br>' +
                         '<extra></extra>'
        ),
        row=1, col=1
    )

# Plot 2: Mean intensity with error bands (mean ± std)
for i, component in enumerate(components):
    color = colors[i % len(colors)]
    mean_vals = np.array(component_means[component])
    std_vals = np.array(component_stds[component])
    
    # Add mean line
    fig.add_trace(
        go.Scatter(
            x=wavelength_values,
            y=mean_vals,
            mode='lines',
            name=f'{component.title()}',
            line=dict(color=color, width=2),
            opacity=0.8,
            showlegend=False,  # Don't show legend for second plot (already shown in first)
            hovertemplate='<b>%{fullData.name}</b><br>' +
                         'Wavelength: %{x:.1f} nm<br>' +
                         'Mean Intensity: %{y:.2f}<br>' +
                         '<extra></extra>'
        ),
        row=1, col=2
    )
    
    # Add upper bound (mean + std)
    fig.add_trace(
        go.Scatter(
            x=wavelength_values,
            y=mean_vals + std_vals,
            mode='lines',
            line=dict(width=0),
            showlegend=False,
            hoverinfo='skip',
            name=f'{component.title()} +std'
        ),
        row=1, col=2
    )
    
    # Add lower bound (mean - std) and fill area
    fig.add_trace(
        go.Scatter(
            x=wavelength_values,
            y=mean_vals - std_vals,
            mode='lines',
            line=dict(width=0),
            fill='tonexty',
            fillcolor=color.replace('rgb', 'rgba').replace(')', ', 0.2)'),  # Add transparency
            showlegend=False,
            hoverinfo='skip',
            name=f'{component.title()} -std'
        ),
        row=1, col=2
    )

# Update layout
fig.update_layout(
    title={
        'text': 'Interactive LIBS Spectral Analysis - Mean Intensities by Component from all tires',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 16}
    },
    width=1400,
    height=600,
    hovermode='x unified',
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02
    )
)

# Update x and y axis labels
fig.update_xaxes(title_text="Wavelength (nm)", row=1, col=1)
fig.update_xaxes(title_text="Wavelength (nm)", row=1, col=2)
fig.update_yaxes(title_text="Mean Intensity", row=1, col=1)
fig.update_yaxes(title_text="Mean Intensity ± Std", row=1, col=2)

# Add grid to both plots
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')

# Show the interactive plot
fig.show()

# Print statistical summary
print(f"\n📊 STATISTICAL SUMMARY BY COMPONENT:")
print(f"{'Component':<12} {'N_Samples':<10} {'Mean_Int':<10} {'Std_Int':<10} {'Max_Int':<10} {'Min_Int':<10}")
print("-" * 70)

for component in components:
    component_data = df[df['origin'] == component]
    mean_vals = np.array(component_means[component])
    
    print(f"{component.title():<12} {len(component_data):<10} {np.mean(mean_vals):<10.1f} "
          f"{np.std(mean_vals):<10.1f} {np.max(mean_vals):<10.1f} {np.min(mean_vals):<10.1f}")



# Component differentiation analysis
print(f"\n🎯 COMPONENT DIFFERENTIATION ANALYSIS:")
for i, comp1 in enumerate(components):
    for j, comp2 in enumerate(components[i+1:], i+1):
        # Calculate mean squared difference between components
        mean1 = np.array(component_means[comp1])
        mean2 = np.array(component_means[comp2])
        mse = np.mean((mean1 - mean2)**2)
        max_diff = np.max(np.abs(mean1 - mean2))
        
        print(f"{comp1.title()} vs {comp2.title()}:")
        print(f"  • Mean Squared Difference: {mse:.2f}")
        print(f"  • Maximum Difference: {max_diff:.2f}")
        
        # Find wavelengths with largest differences
        diff_vals = np.abs(mean1 - mean2)
        top_diff_indices = np.argsort(diff_vals)[-5:]  # Top 5 differences
        top_diff_wavelengths = [wavelength_values[idx] for idx in top_diff_indices]
        
        print(f"  • Top 5 discriminative wavelengths: {[f'{wl:.1f}nm' for wl in top_diff_wavelengths]}")

🔄 Creating interactive mean intensity comparison across all components and wavelengths...



📊 STATISTICAL SUMMARY BY COMPONENT:
Component    N_Samples  Mean_Int   Std_Int    Max_Int    Min_Int   
----------------------------------------------------------------------
Tread        17936      8312.7     5989.7     46116.0    125.9     
Innerliner   17260      11248.3    8245.8     46401.7    125.5     
Sidewall     16257      9266.4     6380.7     39613.0    124.8     

🎯 COMPONENT DIFFERENTIATION ANALYSIS:
Tread vs Innerliner:
  • Mean Squared Difference: 15762380.07
  • Maximum Difference: 11370.54
  • Top 5 discriminative wavelengths: ['766.8nm', '770.1nm', '769.9nm', '766.7nm', '766.5nm']
Tread vs Sidewall:
  • Mean Squared Difference: 1696053.27
  • Maximum Difference: 6502.95
  • Top 5 discriminative wavelengths: ['766.7nm', '777.3nm', '777.7nm', '777.4nm', '777.5nm']
Innerliner vs Sidewall:
  • Mean Squared Difference: 8238557.17
  • Maximum Difference: 9043.11
  • Top 5 discriminative wavelengths: ['656.1nm', '656.7nm', '656.5nm', '656.2nm', '656.4nm']


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   element_origin  140 non-null    object 
 1   wavelength      140 non-null    float64
 2   intensity       140 non-null    float64
dtypes: float64(2), object(1)
memory usage: 3.4+ KB


,element_origin,wavelength,intensity
0,Aluminium,309.3,2.914000e+09
1,Aluminium,309.2,2.781750e+09
2,Aluminium,309.4,2.464200e+09
3,Aluminium,309.1,2.163000e+09
4,Aluminium,186.2,2.108800e+09


convert data below to table format:
Testing different numbers of clusters... 

| N_Clusters | AIC | BIC | Log_Likelihood | Silhouette | ARI | NMI
| ----------- | --------- | --------- | -------------- | ---------- | --- | ---
| 2 | 5200861.0 | 5204940.1 | -50.531 | 0.210 | 0.052 | 0.057
| 3 | 4963539.2 | 4969662.4 | -48.220 | 0.150 | 0.065 | 0.074
| 4 | 4783028.5 | 4791195.5 | -46.462 | 0.124 | 0.140 | 0.134
| 5 | 4647570.2 | 4657781.3 | -45.141 | 0.120 | 0.192 | 0.235
| 6 | 4556204.2 | 4568459.3 | -44.248 | 0.089 | 0.153 | 0.208
| 7 | 4503647.3 | 4517946.3 | -43.733 | 0.071 | 0.114 | 0.172
| 8 | 4458704.6 | 4475047.7 | -43.292 | 0.070 | 0.119 | 0.178
| 9 | 4412585.3 | 4430972.4 | -42.839 | 0.059 | 0.089 | 0.148
| 10 | 4356558.8 | 4376989.8 | -42.290 | 0.078 | 0.101 | 0.185


Uit deze resultaten is op te merken dat ook DBSCAN geen goede clusterresultaten oplevert.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.ndimage import uniform_filter1d
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from scipy import sparse
from scipy.sparse.linalg import spsolve

# --- Synthetisch LIBS-spectrum ---
np.random.seed(42)
x = np.linspace(200, 800, 2000)  # golflengte-as (nm)
baseline_true = 0.0015 * (x - 200) ** 1.2 + 3 * np.sin(x / 120)  # niet-lineaire baseline (synthetisch)
peaks = (
    80 * np.exp(-(x - 300) ** 2 / (2 * 1.5 ** 2))
    + 60 * np.exp(-(x - 420) ** 2 / (2 * 3 ** 2))
    + 90 * np.exp(-(x - 520) ** 2 / (2 * 2 ** 2))
    + 50 * np.exp(-(x - 700) ** 2 / (2 * 4 ** 2))
)
noise = np.random.normal(0, 2, x.size)
y = baseline_true + peaks + noise

# --- Asymmetric Least Squares (AsLS) met sparse matrices ---
def asls_sparse(y, lam=1e5, p=0.001, niter=20):
    L = len(y)
    # maak D^T D bandmatrix (kies de vorm die overeenkomt met tweede differentie)
    e = np.ones(L)
    # bouw een tridiagonale benadering voor stabiliteit; we gebruiken 'banded' opstelling via diag offsets:
    offsets = np.array([0, 1, 2, -1, -2])
    # coëfficiënten (constructie equivalent aan D.T @ D * lam + I*weights)
    diag_main = np.ones(L) * (1 + lam * 6)
    diag_off1 = np.ones(L - 1) * (-4 * lam)
    diag_off2 = np.ones(L - 2) * (lam)
    DTD = sparse.diags([diag_off2, diag_off1, diag_main, diag_off1, diag_off2],
                       offsets=[-2, -1, 0, 1, 2], shape=(L, L), format='csc')
    w = np.ones(L)
    for i in range(niter):
        W = sparse.diags(w, 0, shape=(L, L), format='csc')
        Z = W + DTD
        z = spsolve(Z, w * y)
        w = p * (y > z) + (1 - p) * (y < z)
    return z

# --- Rolling ball (eenvoudige benadering via moving average/uniform filter) ---
def rolling_ball(y, window=101):
    # window moet oneven zijn voor symmetrie
    return uniform_filter1d(y, size=window, mode='mirror')

# --- 4S Peak Filling + AsLS (vereenvoudigd/hybride) ---
def fourS_peak_filling_asls(y, window=41):
    smooth = savgol_filter(y, window_length=window, polyorder=3)
    baseline = asls_sparse(smooth, lam=1e4, p=0.01, niter=15)
    return baseline

# --- Enhanced Savitzky-Golay (iteratief minimum) ---
def enhanced_savgol(y, window=51, poly=3, iterations=5):
    baseline = y.copy()
    for _ in range(iterations):
        smooth = savgol_filter(baseline, window_length=window, polyorder=poly)
        baseline = np.minimum(baseline, smooth)
    return savgol_filter(baseline, window_length=window, polyorder=poly)

# --- Polynomiale fit ---
def poly_baseline(x, y, degree=4):
    poly = PolynomialFeatures(degree)
    X_poly = poly.fit_transform(x.reshape(-1, 1))
    model = LinearRegression().fit(X_poly, y)
    baseline = model.predict(X_poly)
    return baseline

# --- Lineaire fit ---
def linear_baseline(x, y):
    model = LinearRegression().fit(x.reshape(-1, 1), y)
    baseline = model.predict(x.reshape(-1, 1))
    return baseline

# --- Bereken baselines ---
baseline_asls = asls_sparse(y, lam=1e5, p=0.001, niter=25)
baseline_rb = rolling_ball(y, window=121)
baseline_4s = fourS_peak_filling_asls(y, window=41)
baseline_esg = enhanced_savgol(y, window=51, poly=3, iterations=6)
baseline_poly = poly_baseline(x, y, degree=4)
baseline_lin = linear_baseline(x, y)

# --- Plotfunctie (elke plot apart) ---
def plot_baseline(title, baseline, show_true_baseline=True):
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)
    
    # --- Linker paneel: Origineel + baseline ---
    axs[0].plot(x, y, color="black", alpha=0.6, label="Origineel spectrum")
    axs[0].plot(x, baseline, color="red", linewidth=2, label="Geschatte baseline")
    if show_true_baseline:
        axs[0].plot(x, baseline_true, color="green", linestyle="--", linewidth=1.5,
                    label="Werkelijke baseline")
    axs[0].set_title(f"{title}\nVoor correctie")
    axs[0].set_xlabel("Golflengte (nm)")
    axs[0].set_ylabel("Intensiteit (a.u.)")
    axs[0].legend(loc="upper right", fontsize=8)
    axs[0].grid(alpha=0.3)

    # --- Rechter paneel: Gecorrigeerd spectrum ---
    axs[1].plot(x, y - baseline, color="navy", linewidth=1.5, label="Na correctie")
    axs[1].set_title("Na baseline-correctie")
    axs[1].set_xlabel("Golflengte (nm)")
    axs[1].legend(loc="upper right", fontsize=8)
    axs[1].grid(alpha=0.3)

    plt.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


# --- Genereer 6 losse figuren ---
plot_baseline("Asymmetric Least Squares (AsLS)", baseline_asls)
plot_baseline("Rolling Ball (approx.)", baseline_rb)
plot_baseline("4S Peak Filling + AsLS (vereenvoudigd)", baseline_4s)
plot_baseline("Enhanced Savitzky–Golay (iteratief)", baseline_esg)
plot_baseline("Polynomial fit (4e graad)", baseline_poly)
plot_baseline("Lineaire baseline", baseline_lin)

# --- Bewaar gecorrigeerde spectra in een dictionary (optioneel verder opslaan) ---
corrected_spectra = {
    "asls": y - baseline_asls,
    "rolling_ball": y - baseline_rb,
    "4s_asls": y - baseline_4s,
    "enhanced_savgol": y - baseline_esg,
    "poly4": y - baseline_poly,
    "linear": y - baseline_lin
}

print("Klaar — 6 figuren gegenereerd en gecorrigeerde spectra beschikbaar in 'corrected_spectra'.")
